# 04 – Daten passend zum Einsatz aufteilen

**Lernziele:** Testdaten zurückhalten und die Aufteilung aus der Vorhersagefrage ableiten. Dauer: ca. 30 Minuten. Alle Daten werden synthetisch erzeugt und funktionieren offline.

Training lernt Parameter; Validierung unterstützt Auswahlentscheidungen; der Test bewertet am Ende auf unberührten Daten. Erst aufteilen, dann Imputation, Skalierung und Merkmalsauswahl nur im Training lernen.

| Einsatzfrage | Beispiel | Trennung |
|---|---|---|
| Neue unabhängige Einzelbeobachtung | simulierte Bauteilmaße | zufällig |
| Unabhängige Beobachtung, seltene Klasse | Qualitätsprüfung | stratifiziert |
| Neue Maschine / neue Person | mehrere Messungen pro Maschine | Gruppen |
| Zukünftiger Messwert | stündliche Sensorwerte | chronologisch |

Diese Fälle sind Annahmen über den Einsatz. Bei Häusern können beispielsweise Region und Verkaufsjahr eine zufällige Trennung unpassend machen.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, GroupShuffleSplit, TimeSeriesSplit


## 1. Unabhängige Regression: 60 / 20 / 20

20 % Test zurückhalten, dann 25 % der verbleibenden 80 % als Validierung: 0,25 × 0,8 = 0,2. Ein fester Seed macht den Split reproduzierbar, beseitigt aber keine Abhängigkeiten.


In [ ]:
X, y = make_regression(n_samples=600, n_features=3, noise=15, random_state=42)
entwicklung, test = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42)
train, validierung = train_test_split(entwicklung, test_size=0.25, random_state=42)
display(pd.Series({'Training': len(train), 'Validierung': len(validierung), 'Test': len(test)}))
assert not (set(train) & set(validierung) or set(train) & set(test) or set(validierung) & set(test))


## 2. Klassifikation: Klassenanteile erhalten

`stratify=y` erhält näherungsweise die Zielverteilung, gleicht aber die Klassen nicht aus. Sehr kleine Klassen reichen möglicherweise nicht für alle Teilmengen. Bei Regression nicht einfach stetige Zielwerte als `stratify` verwenden.


In [ ]:
X_k, y_k = make_classification(n_samples=1000, n_features=6, weights=[0.95, 0.05], random_state=42)
tr, te = train_test_split(np.arange(len(y_k)), test_size=0.2, stratify=y_k, random_state=42)
display(pd.DataFrame({name: pd.Series(y_k[ix]).value_counts(normalize=True)
                      for name, ix in {'gesamt': np.arange(len(y_k)), 'Training': tr, 'Test': te}.items()}))


## 3. Gruppen: unbekannte Maschinen testen

Messungen derselben Maschine ähneln sich. Ein zufälliger Zeilensplit würde Maschineninformationen über die Grenze verteilen. `test_size=0.2` bei `GroupShuffleSplit` bezieht sich auf Gruppen, nicht allgemein auf die Zeilenzahl.


In [ ]:
rng = np.random.default_rng(42)
maschinen = np.repeat(np.arange(30), 12)
sensor = pd.DataFrame({'maschine': maschinen,
                      'vibration': np.repeat(rng.normal(0, 2, 30), 12) + rng.normal(0, 0.2, 360)})


In [ ]:
zufall_train, zufall_test = train_test_split(np.arange(len(sensor)), test_size=0.2, random_state=42)
print('Maschinen auf beiden Seiten beim Zeilensplit:', len(set(maschinen[zufall_train]) & set(maschinen[zufall_test])))
gruppe_train, gruppe_test = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(sensor, groups=maschinen))
assert set(maschinen[gruppe_train]).isdisjoint(maschinen[gruppe_test])
print('Gruppensplit:', len(set(maschinen[gruppe_train])), 'Trainingsmaschinen;', len(set(maschinen[gruppe_test])), 'Testmaschinen')


## 4. Zeitreihe: Vergangenheit → Zukunft

Wir prognostizieren den nächsten Stundenwert. Features dürfen nur bereits bekannte Werte enthalten: sortieren, Lags aus der Vergangenheit bilden, chronologisch teilen. Kein Shuffle. Bei längeren Horizonten oder überlappenden Fenstern kann ein zusätzlicher Abstand nötig sein.


In [ ]:
zeit = pd.date_range('2025-01-01', periods=240, freq='h')
werte = 20 + 0.03*np.arange(240) + np.sin(np.arange(240)*2*np.pi/24) + rng.normal(0, 0.3, 240)
reihe = pd.DataFrame({'zeit': zeit, 'wert': werte}).sort_values('zeit')
reihe['vorstunde'] = reihe['wert'].shift(1)
reihe = reihe.dropna().reset_index(drop=True)
grenze = int(0.8*len(reihe))
zeit_train, zeit_test = reihe.iloc[:grenze], reihe.iloc[grenze:]
assert zeit_train['zeit'].max() < zeit_test['zeit'].min()
plt.plot(reihe['zeit'], reihe['wert'])
plt.axvline(zeit_test['zeit'].min(), color='red', label='Beginn Test')
plt.ylabel('Sensorwert'); plt.legend(); plt.xticks(rotation=30); plt.tight_layout(); plt.show()


Der Vorstundenwert im späteren Test stammt teilweise aus vorherigen Teststunden. Das ist für laufende Ein-Schritt-Prognosen nach Eingang jeder Messung zulässig. Für die Prognose des gesamten Testzeitraums auf einmal wäre es unzulässig.

**Mini-Aufgabe:** Bilder desselben Bauteils, wiederholte Patientenbesuche, zukünftige Umsätze: Welche Trennung passt jeweils? Bei Gruppen **und** Zeit genügt kein beliebiger Standard-Splitter allein. Bei CWRU aus Set 3 müsste nach Aufnahme/Lager getrennt werden; nur eine Aufnahme je Zustand reicht dafür nicht aus.

Quelle: [scikit-learn: Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html). Weiter mit Notebook 05 und Übung 04.
